# SuttaPlayer Piper1 VITS Training Console (v5)
This notebook serves as your interactive control panel for training the non-rhotic Australian male Sutta voice model on Google Colab's Free Tier T4 GPU. This pipeline utilizes an offline Deno orchestrator, compiles the Cython Monotonic Alignment Search (MAS) C-extension natively, manages an isolated Python 3.11.9 environment (aligned with modern PyTorch 2.x and PyTorch Lightning 2.x), handles background daemons under TMUX, and runs active validation probes with 100% pre-phonemized targets (fully stripped of old carrier signal pads and calibrated against Audacity).

**New in v5:** Integrated offline-first Google Sheets synchronization which processes and appends epoch convergence metrics in real-time, plotting an auto-updating progress-to-target chart entirely decoupled from PyTorch's execution thread!

## Step 1: Mount Google Drive
Mount your Google Drive to expose your training corpus to the Colab container.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2: Provision Deno Runtime
Download and install the Deno security-isolated environment to run the training manager.

In [ ]:
!curl -fsSL https://deno.land/install.sh | sh
import os
os.environ['PATH'] += ':/root/.deno/bin'
!deno --version

## Step 3: Run Environment Initialization
Execute the Deno training manager script. This will automatically:
1. Download and compile **Micromamba**.
2. Provision an isolated **Python 3.11.9** environment.
3. Clone `OHF-Voice/piper1-gpl` and install dependencies (`torch==2.3.1`, `onnx==1.15.0`, `lightning==2.3.3`, `torchaudio==2.3.1` with explicit Setuptools locks).
4. **Compile the Cython Monotonic Alignment Search (MAS) C-Extension** natively inside the virtual environment for 10x faster training loops.
5. Prepare local fast-SSD cache paths under `/content/piper_cache` to shield Google Drive from I/O bottlenecks.

In [ ]:
# Copy the training manager from your Drive base to Colab space and run the init pipeline
!cp /content/drive/MyDrive/piper_training/sutta-training-manager-v7.ts /content/sutta-training-manager-v7.ts
!deno run --allow-all /content/sutta-training-manager-v7.ts --init

## Step 4: Start TensorBoard (Lightweight progress monitoring)
Launch TensorBoard to play synthesized probe WAV files from validation epochs and track global scalar loss curves.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/piper_cache/lightning_logs

## Step 5: Start Training inside TMUX Daemon
Starts the training thread inside a secure TMUX terminal. It integrates `train_sutta_voice.py` as an active callback.

In [ ]:
!deno run --allow-all /content/sutta-training-manager-v7.ts --train

## Step 6: Start Real-Time Synchronizer & Keep-Alive Heartbeat
Runs the real-time background sync daemon. This will:
1. Sync checkpoints (`.ckpt`) back to Google Drive every 10 seconds (rsync).
2. Parse the training execution metrics (`metrics.csv`) from version logs.
3. Issue keep-alive pings to prevent the Google Colab session from timing out or disconnecting.

In [ ]:
!deno run --allow-all /content/sutta-training-manager-v7.ts --monitor

## Step 7: Secure Google Sheets Real-Time Convergence Sync
Run this cell concurrently to authenticate with your Google Account, read the raw `uat_metrics.csv` files generated locally on Google Drive, and push them directly as rows to a synchronized Google Sheet with zero training latency!

In [ ]:
# Authenticate and start Sheets Sync Loop
import os
import time
import pandas as pd
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)

csv_path = "/content/drive/MyDrive/piper_training/uat_metrics.csv"
sheet_name = "SuttaPlayer_UAT_Convergence"

print("🔍 Initializing Google Sheets UAT Sync...")
try:
    try:
        sh = gc.open(sheet_name)
    except gspread.exceptions.SpreadsheetNotFound:
        sh = gc.create(sheet_name)
        print(f"✅ Created new Google Sheet: '{sheet_name}' in your Drive.")
        
    ws = sh.get_worksheet(0)
    
    last_logged_row = len(ws.col_values(1)) if ws.col_values(1) else 0
    print(f"⚡ Active Sync daemon running. Monitoring CSV changes... (Last row logged: {last_logged_row})")
    
    while True:
        if os.path.exists(csv_path):
            try:
                df = pd.read_csv(csv_path)
                total_rows = len(df)
                
                if total_rows > (last_logged_row - 1 if last_logged_row > 0 else 0):
                    start_idx = max(0, last_logged_row - 1 if last_logged_row > 0 else 0)
                    new_data = df.iloc[start_idx:].values.tolist()
                    
                    if last_logged_row == 0:
                        ws.append_row(df.columns.tolist())
                        last_logged_row += 1
                        
                    for row in new_data:
                        ws.append_row(row)
                        print(f"  [Sheets Sync] Appended Epoch {row[1]} | Loss: {row[2]} to Sheet.")
                        
                    last_logged_row = total_rows + 1
            except Exception as e:
                print(f"  [WARNING] Read/write collision on CSV (training is writing). Retrying next tick. Error: {e}")
        time.sleep(15)
except KeyboardInterrupt:
    print("\n⏹️ Sheets Sync Stopped.")